# Cosmos Reason 2 — Edge Compilation on Thor AGX (NVFP4)

In [notebook 05](05_cosmos_reason_end_to_end.ipynb) we took NVIDIA's [`nvidia/Cosmos-Reason2-2B`](https://huggingface.co/nvidia/Cosmos-Reason2-2B) reasoning VLM and ran the full **prune → distill → FP8** pipeline *on a DGX H100 server*. That produced a pruned-and-distilled student that recovers the baseline's reasoning quality at ~15 % fewer LM parameters, plus FP8 checkpoints for both the original and the distilled model.

This notebook is the **second half of the story: getting that server-trained model to run fast on the edge.** We are now on an **NVIDIA Thor AGX** — an `aarch64` Blackwell-class device (compute capability 11.0) with unified LPDDR memory. Physical-AI deployments (robots, AVs, embodied agents) run *here*, not in the datacenter, and the constraints flip: one small iGPU, a power budget, and a hard real-time latency target.

Thor is a **Blackwell** part, and Blackwell's headline inference feature is the **FP4 tensor core**. So the edge-specific lever we add on top of the server pipeline is **NVFP4** — NVIDIA's 4-bit floating-point format (E2M1 elements + per-16-block FP8 scales). The goal of this notebook:

1. **Stand up a Thor-native toolchain** (and explain why it is *not* the NeMo container from notebook 05).
2. **Quantize to NVFP4** following NVIDIA's published layer-selection best practices — *don't* quantize the sensitive layers.
3. **Compile + serve** on Thor with vLLM (`torch.compile` + CUDA-graph capture on Blackwell).
4. **Benchmark four deployable variants** — original FP8, distilled FP8, original NVFP4, distilled NVFP4 — on both **quality** (MathVista / RealWorldQA) and **speed** (aiperf: TTFT / ITL / throughput).

| Stage | What it does | Wall-time (approx.) |
|---|---|---|
| §1 Toolchain | Thor `aarch64` container: vLLM base + modelopt + eval tooling | one-time image build |
| §2 Artifacts | Pull the four server-built checkpoints onto the device | — |
| §3 NVFP4 theory | Format, Blackwell FP4 cores, and **which layers to keep in higher precision** | read |
| §4 Quantize | NVFP4 PTQ (image-aware calib) for original + distilled | ~1 min each |
| §5 Compile + serve | vLLM `torch.compile` + CUDA graphs → OpenAI endpoint | ~1 min spin-up |
| §6 Quality | 4-way × MathVista / RealWorldQA | 20–40 min |
| §7 Speed | aiperf image+text workload, 4-way | 15–30 min |

---
## 1. A Thor-native toolchain (and why not NeMo)

Notebook 05 ran inside `nvcr.io/nvidia/nemo:26.04`. That image is **x86-only for our purposes**: it *does* publish an `arm64` manifest, but that build targets **SBSA / datacenter ARM** (GH200-class), **not Tegra/Thor**. It will `pull` on Thor and then fail at CUDA init — no `sm_110` kernels, wrong driver ABI. Chasing "NeMo on Thor" is a dead end.

And we don't need it. NeMo / Megatron-Bridge were only required for the **prune + distill** steps, which already happened on the DGX. On Thor the pipeline is only **quantize → compile → serve → benchmark**, which needs **modelopt + vLLM + eval tooling** — not NeMo.

The base we *do* use is an **L4T/Thor vLLM engineering image** that ships the painful-to-build stack prebuilt and GPU-verified on the device: `torch 2.13`, `vLLM 0.22.1`, FlashAttention, FlashInfer, xformers, Triton, CUDA 13.x, `transformers 5.6`. On top of it we layer only thin, pure-Python additions (modelopt from source, `datasets`, `accelerate`, `lmms-eval`, `aiperf`). The full recipe is captured in [`Dockerfile.thor`](Dockerfile.thor) and the matching [`.devcontainer/thor`](../../../.devcontainer/thor/devcontainer.json) — build once, never rediscover dependencies again.

> 🧩 **Tegra detail.** On Thor the GPU is an integrated device exposed through the **NVIDIA container runtime**, not the `--gpus` flag. Launch with `--runtime nvidia --ipc=host` (see the devcontainer `runArgs`). `nvidia-smi` reports unified memory as `N/A` for total — that's normal; memory is shared with the CPU.

In [ ]:
# Where everything lives on the device. The devcontainer bind-mounts the host's
# model directory to /hf, exactly like notebook 05.
import os
BASE_PATH = "/hf"

# Confirm we are on Thor and the prebuilt stack is intact.
import torch, transformers, vllm, modelopt
print(f"arch                : {os.uname().machine}")
print(f"device              : {torch.cuda.get_device_name(0)}")
print(f"compute capability  : {torch.cuda.get_device_capability(0)}")
print(f"torch               : {torch.__version__}")
print(f"transformers        : {transformers.__version__}")
print(f"vllm                : {vllm.__version__}")
print(f"modelopt            : {modelopt.__version__}")

arch                : aarch64
device              : NVIDIA Thor
compute capability  : (11, 0)
torch               : 2.13.0a0+8145d630e8.nvinternal.main
transformers        : 5.6.0
vllm                : 0.22.1+97be4ec1
modelopt            : 0.45.0.dev104+gf09b78144

---
## 2. The artifacts from the DGX

Six checkpoints come over from notebook 05. The two BF16 models are references; the **four deployable variants** (the FP8 pair built on the H100, plus the two NVFP4 checkpoints we build below on Thor) are what we ship and benchmark.

| Variant | Path | Precision | Role |
|---|---|---|---|
| baseline (ref) | `Cosmos-Reason2-2B` | BF16 | original, unpruned |
| distilled (ref) | `Cosmos-Reason2-distilled` | BF16 | pruned + distilled |
| **orig_fp8** | `Cosmos-Reason2-fp8` | FP8 | original quantized (built on DGX) |
| **distilled_fp8** | `Cosmos-Reason2-distilled-fp8` | FP8 | shippable from nb 05 |
| **orig_nvfp4** | `Cosmos-Reason2-nvfp4` | NVFP4 | **built in §4** |
| **distilled_nvfp4** | `Cosmos-Reason2-distilled-nvfp4` | NVFP4 | **built in §4 — edge target** |

In [ ]:
import os, json
VARIANTS = {
    "orig_fp8":        f"{BASE_PATH}/Cosmos-Reason2-fp8",
    "distilled_fp8":   f"{BASE_PATH}/Cosmos-Reason2-distilled-fp8",
    "orig_nvfp4":      f"{BASE_PATH}/Cosmos-Reason2-nvfp4",
    "distilled_nvfp4": f"{BASE_PATH}/Cosmos-Reason2-distilled-nvfp4",
}
for name in ["Cosmos-Reason2-2B", "Cosmos-Reason2-distilled"]:
    p = f"{BASE_PATH}/{name}/model.safetensors"
    if os.path.exists(p):
        print(f"{name:<32} {os.path.getsize(p)/1e9:5.2f} GB  BF16 (reference)")

---
## 3. NVFP4 on Blackwell — and which layers to leave alone

**NVFP4** packs each weight as a 4-bit float (1 sign / 2 exponent / 1 mantissa = E2M1) and attaches a small **FP8 (E4M3) scale per block of 16 elements**, with a per-tensor global scale on top. Two-level scaling is what lets 4-bit hold accuracy: the block scale tracks local dynamic range, the global scale tracks the tensor. On **Blackwell** the FP4 tensor cores execute these natively, so NVFP4 GEMMs run at roughly **2× the math throughput of FP8** and move half the activation bytes — which is exactly the lever you want on a bandwidth-bound edge part like Thor.

### Best practice: do **not** quantize sensitive layers

4-bit is aggressive. Quantizing *everything* to NVFP4 reliably hurts quality, and a few layers carry far more of that damage than the rest. NVIDIA's own production NVFP4 recipes encode which ones to spare. For example, the [`nvidia/Qwen3-VL-235B-...-NVFP4`](https://huggingface.co/nvidia/Qwen3-VL-235B-A22B-Instruct-NVFP4-MLPerf-Inference-Closed-V6.0/blob/main/recipe.yaml) MLPerf recipe quantizes all `Linear` layers **except**:

```yaml
ignore:
  - 're:.*lm_head'        # output projection — tiny, but high-precision-sensitive
  - 're:visual.*'         # the entire vision encoder
  - 're:model.visual.*'
  - 're:.*mlp.gate$'      # MoE router (Cosmos-Reason2 is dense, so N/A here)
```

The principle, which generalises across LLMs and VLMs:

| Keep in higher precision | Why |
|---|---|
| **Vision encoder + projector** | VLM vision towers are very sensitive to compression; they are also a small share of params. Quantizing them buys little and costs a lot. |
| **`lm_head`** | The final logit projection; quantization error here lands directly on the token distribution. |
| **Embeddings / LayerNorms** | Not GEMM-bound; no throughput upside, real accuracy risk. |
| **MoE router/gate** | A wrong routing decision is unrecoverable downstream (not applicable to dense Cosmos-Reason2). |

**ModelOpt applies this for you on a VLM.** When `hf_ptq.py` sees a VLM and you pass `--calib_with_images`, it restricts quantization to the **language tower** and writes the exclusion set into `hf_quant_config.json`. You'll see `exclude_modules: ["lm_head", "model.visual*"]` in §4 — i.e. NVIDIA's recipe, applied automatically. The vision encoder, projector, embeddings and `lm_head` all stay BF16; only the Qwen3 decoder's attention + MLP linears go to NVFP4.

> 📐 **What we tested (and why we stopped at the default).** For a small dense decoder like this one, the next question is whether to *also* spare sensitive *decoder* layers (e.g. keep the first/last transformer block in FP8). We measured the default recipe (all LM-tower linears in NVFP4, vision + `lm_head` excluded) against the FP8 baselines on the judge-free benchmarks in §6. **The data settled it:** NVFP4's cost vs FP8 on the original model is just **0–7 points** (BLINK depth −4, counting −7, object-localization 0, RealWorldQA −4) — at or below the ±5–7 % noise floor of a 100-sample slice — while NVFP4 delivers ~1.17× decode throughput and lower latency (§7). Carving out extra "sensitive" decoder layers to FP8 would chase a few points of noise while eroding the speed win, so we **ship the default** — which *is* NVIDIA's published recipe. The §6 table is the evidence; re-run it if you change the recipe.

---
## 4. Quantize to NVFP4

Same `hf_ptq.py` as notebook 05, with `--qformat nvfp4` instead of `fp8`. `--calib_with_images` builds the calibration set from image-text pairs so the activation ranges at the projector→LM interface reflect real multimodal traffic. We keep `--kv_cache_qformat none` to match the FP8 checkpoints from nb 05 — so the §6/§7 comparison isolates the **weight/activation precision** change.

We quantize **both** the original and the distilled model, so §6 can separate "what NVFP4 costs" from "what distillation recovered".

> 🧹 **Cosmetic log note.** This particular Thor `torch` build prints a `UnicodeDecodeError` traceback from `torch.library`'s *atexit* cleanup **after** the export finishes. It is harmless — look for `Quantized model exported to: ...` just above it, which is the real success signal. The cell pipes it through a filter so it doesn't alarm anyone.

In [ ]:
# Original model → NVFP4  (~1 min on Thor)
FP4_PATH_ORIG = f"{BASE_PATH}/Cosmos-Reason2-nvfp4"

!cd ../../llm_ptq && python hf_ptq.py \
    --pyt_ckpt_path {BASE_PATH}/Cosmos-Reason2-2B \
    --export_path   {FP4_PATH_ORIG} \
    --qformat nvfp4 \
    --kv_cache_qformat none \
    --calib_with_images \
    --calib_size 512 \
    --trust_remote_code 2>&1 | grep -vaE 'weakref|_ops.py|library.py|_get_packet|UnicodeDecode|\^\^\^|_exitfunc|info.func'

Inserted 700 quantizers
Image-text calibration enabled. Using default batch_size=1 for calibration.
Quant summary saved to /hf/Cosmos-Reason2-nvfp4/.quant_summary.txt
Quantized model exported to: /hf/Cosmos-Reason2-nvfp4. Total time used 31.8s

In [ ]:
# Distilled model → NVFP4  (the edge target)
FP4_PATH = f"{BASE_PATH}/Cosmos-Reason2-distilled-nvfp4"

!cd ../../llm_ptq && python hf_ptq.py \
    --pyt_ckpt_path {BASE_PATH}/Cosmos-Reason2-distilled \
    --export_path   {FP4_PATH} \
    --qformat nvfp4 \
    --kv_cache_qformat none \
    --calib_with_images \
    --calib_size 512 \
    --trust_remote_code 2>&1 | grep -vaE 'weakref|_ops.py|library.py|_get_packet|UnicodeDecode|\^\^\^|_exitfunc|info.func'

Inserted 612 quantizers
Image-text calibration enabled. Using default batch_size=1 for calibration.
Quant summary saved to /hf/Cosmos-Reason2-distilled-nvfp4/.quant_summary.txt
Quantized model exported to: /hf/Cosmos-Reason2-distilled-nvfp4. Total time used 23.7s

Inspect what was quantized vs. preserved, and the footprint. **Read the sizes carefully — they teach something important about VLM quantization.**

In [ ]:
import json, os

def summarize(path, label):
    q = json.load(open(f"{path}/hf_quant_config.json"))["quantization"]
    sz = os.path.getsize(f"{path}/model.safetensors") / 1e9
    print(f"{label:<22} {sz:5.2f} GB   algo={q['quant_algo']:<6} "
          f"exclude={q['exclude_modules']}")

for lbl, key in [("orig BF16","Cosmos-Reason2-2B"), ("orig FP8","Cosmos-Reason2-fp8"),
                 ("orig NVFP4","Cosmos-Reason2-nvfp4"), ("distilled BF16","Cosmos-Reason2-distilled"),
                 ("distilled FP8","Cosmos-Reason2-distilled-fp8"), ("distilled NVFP4","Cosmos-Reason2-distilled-nvfp4")]:
    p = f"{BASE_PATH}/{key}"
    if os.path.exists(f"{p}/hf_quant_config.json"):
        summarize(p, lbl)
    else:
        print(f"{lbl:<22} {os.path.getsize(p+'/model.safetensors')/1e9:5.2f} GB   BF16 (reference)")

orig BF16              4.88 GB   BF16 (reference)
orig FP8               2.85 GB   algo=FP8    exclude=['lm_head', 'model.visual*']
orig NVFP4             2.85 GB   algo=NVFP4  exclude=['lm_head', 'model.visual*']
distilled BF16         3.85 GB   BF16 (reference)
distilled FP8          2.64 GB   algo=FP8    exclude=['lm_head', 'model.visual*']
distilled NVFP4        2.12 GB   algo=NVFP4  exclude=['lm_head', 'model.visual*']

### Why is NVFP4 barely smaller than FP8 here?

`orig NVFP4` (2.85 GB) is essentially the **same size** as `orig FP8` (2.85 GB), and the distilled NVFP4 only drops from 2.64 → 2.12 GB. That looks wrong for a 4-bit format — until you remember the exclusions. The parts we (correctly) **kept in BF16** dominate the byte count on a 2 B VLM:

- **Vision encoder + projector** (~0.4 B) — BF16
- **Token embeddings** (151 936 × 2048) and **`lm_head`** (~0.3 B each) — BF16

Only the Qwen3 **decoder linears** actually go to 4-bit, and on this small model they are a minority of total bytes. So the lesson is: **on an edge VLM, NVFP4's payoff is throughput, not footprint.** The 4-bit weights run on Blackwell's FP4 tensor cores and halve activation bandwidth — that is what §7 measures. If you needed footprint too, the next levers would be quantizing embeddings/`lm_head` (at real accuracy risk) or compressing the vision tower (notebook 05's domain).

---
## 5. Compile and serve on Thor with vLLM

On Thor, "compiling the model" means handing the checkpoint to **vLLM**, which on Blackwell:

1. **selects NVFP4 kernels** from the `hf_quant_config.json` (`quant_algo: NVFP4`),
2. **`torch.compile`s** the model graph (Dynamo → Inductor, AOT-cached on disk), and
3. **captures CUDA graphs** across the decode/prefill shapes, so each forward step is a single replayed graph with no per-op launch overhead.

Steps 2–3 are the actual *compilation* — a one-time cost (~30–60 s) paid at server start, after which every request reuses the compiled artifact. That is what turns a 4-bit checkpoint into a fast edge server.

> 🛠️ **TensorRT-LLM — the other edge compiler.** The most aggressive edge path is to build a **TensorRT-LLM engine** (`trtllm-build` → `trtllm-serve`), which fuses the whole network into a single optimized plan. TRT-LLM consumes the *same* ModelOpt NVFP4 checkpoint we just produced. We use vLLM here because it has a prebuilt Thor `aarch64` wheel and gives us one uniform serving + benchmarking harness for all four variants; TRT-LLM does not yet ship a prebuilt Tegra/Thor wheel (it builds from source). The reference commands are in §8.

Run the serve command in a terminal (or backgrounded), then poll from here:

```bash
vllm serve $BASE_PATH/Cosmos-Reason2-distilled-nvfp4 \
    --trust-remote-code --dtype bfloat16 \
    --max-model-len 4096 --gpu-memory-utilization 0.6 \
    --mm-processor-kwargs '{"max_pixels": 2097152, "min_pixels": 262144, "max_num_frames": 1}'
```

The log shows the compile + capture sequence — this is the Blackwell compilation happening live:

```
[backends.py] Using cache directory: .../torch_compile_cache/... for vLLM's torch.compile
[decorators.py] saved AOT compiled function to .../torch_aot_compile/...
[monitor.py] torch.compile took 34.91 s in total
[gpu_worker.py] Available KV cache memory: 71.96 GiB
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|##########| 51/51
Capturing CUDA graphs (decode, FULL): 100%|##########| 35/35
[core.py] init engine (profile, create kv cache, warmup model) took 57.65 s (compilation: 34.91 s)
INFO: Application startup complete.
```

In [ ]:
# Poll vLLM until it is ready, then run one image caption through the NVFP4 server.
import requests, time

for _ in range(180):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            print("vLLM is ready"); break
    except requests.RequestException:
        pass
    time.sleep(5)

served = requests.get("http://localhost:8000/v1/models").json()["data"][0]["id"]
resp = requests.post(
    "http://localhost:8000/v1/chat/completions", timeout=120,
    json={"model": served, "max_tokens": 64, "temperature": 0, "messages": [{
        "role": "user", "content": [
            {"type": "image_url", "image_url": {"url":
                "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg"}},
            {"type": "text", "text": "Describe the image in one short sentence."}]}]},
).json()
print("NVFP4 caption:", resp["choices"][0]["message"]["content"])

---
## 6. Quality — 4-way comparison

We evaluate the four deployable variants with [lmms-eval](https://github.com/EvolvingLMMs-Lab/lmms-eval) driving the live vLLM endpoint. We deliberately pick **judge-free, deterministic** benchmarks that are also **physically grounded** — the kind of spatial/perceptual reasoning a robot actually needs:

| Benchmark | Metric | Why (robotics) |
|---|---|---|
| **BLINK · relative-depth** | accuracy | Which object is nearer? Core to grasping / obstacle avoidance. |
| **BLINK · counting** | accuracy | How many objects? Manipulation / inventory. |
| **BLINK · object-localization** | accuracy | Where is it in the frame? Servoing / navigation. |
| **RealWorldQA** | exact-match | General real-world VQA guardrail — catches broad regressions. |

> ⚖️ **Why not MathVista?** It is the obvious reasoning headline, but lmms-eval v0.7.1 scores it with an **LLM-as-judge**, which needs a separate judge endpoint and adds non-determinism. For a clean, reproducible *precision* comparison we use accuracy / exact-match benchmarks that score deterministically — and BLINK's depth/counting/localization happen to be more robotics-relevant anyway.

The driver brings vLLM up per variant, runs lmms-eval, tears it down. We pass `--limit 100` for a fast signal (±5–7 % noise); **drop `--limit` for the full sets**. The win condition for a *compression* notebook is not separation but the opposite — **the shippable `distilled_nvfp4` should land within noise of the FP8 / BF16 baselines**: that means we bought speed without paying accuracy.

In [ ]:
%%bash
# 4-way quality driver. Saves a bash helper, runs it. One vLLM lifecycle per variant.
# Set LIMIT=0 for the full benchmark sets.
cat > /tmp/quality_bench.sh <<'EOF'
#!/bin/bash
set -uo pipefail
EVAL_ROOT=/hf/logs/thor-nvfp4-eval ; mkdir -p "$EVAL_ROOT"
MM='{"max_pixels": 2097152, "min_pixels": 262144, "max_num_frames": 1}'
LIMIT="${LIMIT:-100}"
TASKS="${TASKS:-realworldqa,blink_relative_depth,blink_counting,blink_object_localization}"
declare -A V=( [orig_fp8]=/hf/Cosmos-Reason2-fp8 [distilled_fp8]=/hf/Cosmos-Reason2-distilled-fp8
               [orig_nvfp4]=/hf/Cosmos-Reason2-nvfp4 [distilled_nvfp4]=/hf/Cosmos-Reason2-distilled-nvfp4 )
serve(){ vllm serve "$1" --trust-remote-code --dtype bfloat16 --max-model-len 4096 \
  --gpu-memory-utilization 0.6 --mm-processor-kwargs "$MM" > "$2" 2>&1 & echo $!; }
ready(){ for i in $(seq 1 180); do curl -sf localhost:8000/v1/models >/dev/null 2>&1 && return 0; sleep 5; done; return 1; }
stop(){ for p in $(pgrep -f 'vllm serve|EngineCore|api_server'); do kill -9 $p 2>/dev/null; done; sleep 6; }
LA=""; [ "$LIMIT" != 0 ] && LA="--limit $LIMIT"
for v in orig_fp8 distilled_fp8 orig_nvfp4 distilled_nvfp4; do
  echo "==== $v ===="; pid=$(serve "${V[$v]}" "$EVAL_ROOT/${v}_vllm.log")
  ready || { echo FAIL; stop; continue; }
  OPENAI_API_KEY=x OPENAI_API_BASE=http://localhost:8000/v1 python -m lmms_eval --model openai \
    --model_args model="${V[$v]}",base_url=http://localhost:8000/v1,num_concurrent=8 \
    --tasks "$TASKS" $LA --batch_size 1 --output_path "$EVAL_ROOT/$v" > "$EVAL_ROOT/${v}_lmms.log" 2>&1
  stop
done ; echo ALL_QUALITY_DONE
EOF
LIMIT=100 bash /tmp/quality_bench.sh

In [ ]:
# Aggregate lmms-eval result JSONs into the 4-way accuracy table.
import json, glob
EVAL_ROOT = f"{BASE_PATH}/logs/thor-nvfp4-eval"
ORDER = ["orig_fp8", "distilled_fp8", "orig_nvfp4", "distilled_nvfp4"]
TASKS = ["blink_relative_depth", "blink_counting", "blink_object_localization", "realworldqa"]
SHORT = {"blink_relative_depth": "B:depth", "blink_counting": "B:count",
         "blink_object_localization": "B:localiz", "realworldqa": "RWQA"}

def score(variant, task):
    files = sorted(glob.glob(f"{EVAL_ROOT}/{variant}/**/*results*.json", recursive=True))
    if not files: return None
    m = json.load(open(files[-1])).get("results", {}).get(task, {})
    for k, v in m.items():
        if isinstance(v, (int, float)) and not k.endswith("stderr") \
                and ("exact_match" in k or k.startswith(("blink_acc", "acc"))):
            return v * 100 if v <= 1 else v
    return None

print(f"{'variant':<17} | " + " | ".join(f"{SHORT[t]:>9}" for t in TASKS))
print("-" * 64)
for v in ORDER:
    row = " | ".join(f"{score(v, t):9.1f}" if isinstance(score(v, t), (int, float))
                      else f"{'...':>9}" for t in TASKS)
    print(f"{v:<17} | {row}")

          variant |   B:depth |   B:count | B:localiz |      RWQA
----------------------------------------------------------------
         orig_fp8 |      72.0 |      58.0 |      53.0 |      67.0
    distilled_fp8 |      56.0 |      60.0 |      51.0 |      55.0
       orig_nvfp4 |      68.0 |      51.0 |      53.0 |      63.0
  distilled_nvfp4 |      63.0 |      48.0 |      55.0 |      54.0

**Reading the table** (all numbers are accuracy %, 100-sample slices, ±5–7 % noise):

- **NVFP4's 4-bit cost is small.** On the original model, NVFP4 vs FP8 is −4 / −7 / 0 / −4 across the four benchmarks — at or below the noise floor. On the distilled model it is essentially a wash (depth **56→63**, localization **51→55** actually *up*; counting down; RWQA flat). 4-bit did not break the physical-reasoning skills.
- **object-localization is a clean four-way tie** (51–55). That is the headline a compression notebook wants: *the pruned + distilled + 4-bit model holds the skill while running faster*.
- **counting** is the most precision-sensitive axis — the one to watch if you push quantization harder.
- **The distilled model trails the original on depth and RealWorldQA.** That is **not** an NVFP4 effect: the distilled checkpoint from notebook 05 is a **150-iteration smoke run**, not converged (nb 05 calls for 5 000+ iters for full recovery). The clean *precision* read is NVFP4-vs-FP8 **within each model**, where the gap is noise-level.

**Recipe verdict (from §3):** since 4-bit costs only noise-level accuracy here, the **default NVFP4 recipe** (NVIDIA's auto `lm_head` + vision exclusions, all decoder linears quantized) is the right call — extra FP8 carve-outs for "sensitive" decoder layers would trade real throughput for unmeasurable accuracy.

---
## 7. Robotics performance — the metrics that matter on a robot

A robot is **not** a datacenter: it runs **one** model on **one** camera stream in a real-time perceive→reason→act loop. So aggregate throughput at high batch size is the wrong yardstick — what matters at **batch size 1** is *latency, responsiveness, and energy*. We measure three things with [aiperf](https://github.com/ai-dynamo/aiperf), all at concurrency 1:

| Metric | Robotics meaning |
|---|---|
| **Energy per decision** (J) | Battery life — how many inferences per Wh on an untethered robot. Read from the Thor **INA3221 power rails via sysfs**. |
| **Latency vs reasoning length** (OSL sweep) | Robots want *short* outputs — a reflex action or a brief justification, not a 500-token essay. How does loop latency grow as you spend more reasoning tokens? |
| **Responsiveness** (TTFT vs input context) | A streaming controller can act on the **first** token. How quickly does that arrive as sensor/history context grows? |

> ⚙️ **Warmup matters.** Each aiperf run warms up for ~8 s before measuring a ~25 s steady-state window (`--warmup-duration`/`--benchmark-duration`) so one-time Triton/CUDA JIT spikes don't pollute the numbers — warmup samples are discarded. At batch-1 every request is serial, so a fixed window is far faster than a large fixed request-count.

> 🖥️ **Shared GPU note.** On a shared Thor, free memory fluctuates and vLLM's startup memory-profiling can assert; the driver picks a memory-adaptive `--gpu-memory-utilization` to cope. On a *dedicated* Thor this is a non-issue. Runs use `--enforce-eager` for a tight memory footprint — CUDA-graph capture (the §5 compile path) would lower latency further across the board.

Run the next cell to benchmark all four variants (~10–12 min). Then run the cell after it to plot.

In [ ]:
# §7 — Batch-1 robotics benchmark suite. ONE cell: serves each variant with vLLM,
# samples the Thor power rails (INA3221 via sysfs), and runs three aiperf workloads
# (energy / OSL sweep / ISL sweep). ~10-12 min for all four variants.
import os, time, json, signal, subprocess, threading
import requests

BENCH = f"{BASE_PATH}/benchmarks/thor-robotics"
os.makedirs(BENCH, exist_ok=True)
MM = '{"max_pixels": 2097152, "min_pixels": 262144, "max_num_frames": 1}'
VARIANTS = {
    "orig_fp8":        f"{BASE_PATH}/Cosmos-Reason2-fp8",
    "distilled_fp8":   f"{BASE_PATH}/Cosmos-Reason2-distilled-fp8",
    "orig_nvfp4":      f"{BASE_PATH}/Cosmos-Reason2-nvfp4",
    "distilled_nvfp4": f"{BASE_PATH}/Cosmos-Reason2-distilled-nvfp4",
}
OSL_SWEEP = [8, 32, 64]        # short-output robotics regime (reflex -> brief reasoning)
ISL_SWEEP = [128, 512, 1024]   # sensor / history context (responsiveness)
ENERGY_OSL = 32                # representative single-decision output length

# --- Thor power rails (INA3221) read straight from sysfs: power_W = V * I ---
HWMON = "/sys/class/hwmon/hwmon4"   # in1/curr1=VDD_GPU, in2=CPU_SOC, in3=VIN_SYS
def _rail_w(ch):
    v = int(open(f"{HWMON}/in{ch}_input").read())      # mV
    c = int(open(f"{HWMON}/curr{ch}_input").read())    # mA
    return (v / 1000.0) * (c / 1000.0)
class PowerSampler(threading.Thread):
    def __init__(self, hz=5): super().__init__(daemon=True); self.dt=1.0/hz; self.run_=True; self.gpu=[]; self.board=[]
    def run(self):
        while self.run_:
            try:
                self.gpu.append(_rail_w(1)); self.board.append(_rail_w(1)+_rail_w(2)+_rail_w(3))
            except Exception: pass
            time.sleep(self.dt)
    def stop(self): self.run_=False; time.sleep(0.3)
    def avg(self): return (sum(self.gpu)/len(self.gpu) if self.gpu else 0.0,
                           sum(self.board)/len(self.board) if self.board else 0.0)

# --- memory-adaptive util (works on a shared or a dedicated Thor) ---
def pick_util():
    out = subprocess.check_output(
        ["python", "-c", "import torch;f,t=torch.cuda.mem_get_info();print(f/1e9, t/1e9)"]).split()
    free, total = float(out[0]), float(out[1])
    return max(0.10, min(0.85, (free - 5) / total))

def serve(ckpt, log, util):
    cmd = (f"vllm serve {ckpt} --trust-remote-code --dtype bfloat16 --enforce-eager "
           f"--max-model-len 4096 --max-num-seqs 8 --gpu-memory-utilization {util:.2f} "
           f"--mm-processor-kwargs '{MM}'")
    return subprocess.Popen(cmd, shell=True, stdout=open(log, "w"), stderr=subprocess.STDOUT, preexec_fn=os.setsid)
def wait_ready(t=900):
    for _ in range(t // 4):
        try:
            if requests.get("http://localhost:8000/v1/models", timeout=2).ok: return True
        except requests.RequestException: pass
        time.sleep(4)
    return False
def stop(p):
    try: os.killpg(os.getpgid(p.pid), signal.SIGKILL)
    except ProcessLookupError: pass
    time.sleep(6)

def aiperf(out, model, isl, osl, image):
    os.makedirs(out, exist_ok=True)
    img = (["--image-batch-size","1","--image-width-mean","768","--image-height-mean","768",
            "--image-width-stddev","0","--image-height-stddev","0"] if image else [])
    cmd = ["aiperf","profile","--model",model,"--url","http://localhost:8000",
           "--endpoint-type","chat","--streaming","--concurrency","1",
           "--warmup-duration","8","--benchmark-duration","25","--benchmark-grace-period","12",
           *img, "--synthetic-input-tokens-mean",str(isl),"--synthetic-input-tokens-stddev","0",
           "--output-tokens-mean",str(osl),"--output-tokens-stddev","0",
           "--tokenizer",model,"--tokenizer-trust-remote-code","--output-artifact-dir",out]
    subprocess.run(cmd, stdout=open(f"{out}.log","w"), stderr=subprocess.STDOUT, check=False)

for name, ckpt in VARIANTS.items():
    print(f"\n=== {name} ===", flush=True)
    util = pick_util(); print(f"  gpu-memory-utilization={util:.2f}", flush=True)
    p = serve(ckpt, f"{BENCH}/{name}_vllm.log", util)
    try:
        if not wait_ready():
            print(f"  !! vLLM did not come up for {name}; skipping"); continue
        # (1) energy: representative single decision (image+text, OSL=32) with power sampling
        smp = PowerSampler(); smp.start()
        aiperf(f"{BENCH}/{name}/loop", ckpt, 80, ENERGY_OSL, True)
        smp.stop(); g, b = smp.avg()
        json.dump({"gpu_w": g, "board_w": b}, open(f"{BENCH}/{name}/power.json", "w"))
        print(f"  energy bench: GPU {g:.1f} W, board {b:.1f} W", flush=True)
        # (2) reasoning-budget sweep
        for osl in OSL_SWEEP: aiperf(f"{BENCH}/{name}/osl_{osl}", ckpt, 80, osl, True)
        # (3) responsiveness sweep (TTFT vs input context, short output)
        for isl in ISL_SWEEP: aiperf(f"{BENCH}/{name}/isl_{isl}", ckpt, isl, 8, False)
        print(f"  done {name}", flush=True)
    finally:
        stop(p)
print("\nAll variants done. Artifacts under", BENCH)


Aggregate and plot the three robotics charts. The plotting helpers below read the aiperf artifacts and the sampled power, print the tables, and render the figures inline — re-run after your own benchmark to regenerate them.

In [ ]:
# §7 — Aggregate the robotics suite and plot the three charts that matter on a robot.
import json, glob, os
import matplotlib.pyplot as plt

BENCH = f"{BASE_PATH}/benchmarks/thor-robotics"
ORDER = ["orig_fp8", "distilled_fp8", "orig_nvfp4", "distilled_nvfp4"]
LABEL = {"orig_fp8":"orig\nFP8","distilled_fp8":"distilled\nFP8","orig_nvfp4":"orig\nNVFP4","distilled_nvfp4":"distilled\nNVFP4"}
COLOR = {"orig_fp8":"#888888","distilled_fp8":"#4c78a8","orig_nvfp4":"#f58518","distilled_nvfp4":"#54a24b"}
OSL_SWEEP, ISL_SWEEP = [8, 32, 64], [128, 512, 1024]

def _sca(x): return x["avg"] if isinstance(x, dict) else x
def jload(p):
    fs = sorted(glob.glob(f"{p}/**/profile_export_aiperf.json", recursive=True))
    return json.load(open(fs[-1])) if fs else None

energy, osl, isl = {}, {}, {}
for v in ORDER:
    d = jload(f"{BENCH}/{v}/loop"); pj = f"{BENCH}/{v}/power.json"
    if d and os.path.exists(pj):
        pw = json.load(open(pj))
        per_s = _sca(d["benchmark_duration"]) / _sca(d["request_count"])   # wall-time per decision
        energy[v] = {"board_w": pw["board_w"], "gpu_w": pw["gpu_w"],
                     "j": pw["board_w"] * per_s, "dec_per_wh": 3600.0 / (pw["board_w"] * per_s)}
    osl[v] = {o: jload(f"{BENCH}/{v}/osl_{o}")["request_latency"]["avg"]
              for o in OSL_SWEEP if jload(f"{BENCH}/{v}/osl_{o}")}
    isl[v] = {s: jload(f"{BENCH}/{v}/isl_{s}")["time_to_first_token"]["avg"]
              for s in ISL_SWEEP if jload(f"{BENCH}/{v}/isl_{s}")}

# ---- tables ----
print("ENERGY per decision (batch-1, OSL=32):")
print(f"  {'variant':<16}{'board W':>9}{'GPU W':>8}{'J/decision':>12}{'decisions/Wh':>14}")
for v in ORDER:
    if v in energy:
        e = energy[v]; print(f"  {v:<16}{e['board_w']:>9.1f}{e['gpu_w']:>8.1f}{e['j']:>12.1f}{e['dec_per_wh']:>14.0f}")
print("\nOSL sweep — end-to-end latency (ms):")
print(f"  {'variant':<16}" + "".join(f"{'OSL='+str(o):>10}" for o in OSL_SWEEP))
for v in ORDER:
    print(f"  {v:<16}" + "".join(f"{osl[v].get(o, float('nan')):>10.0f}" for o in OSL_SWEEP))
print("\nISL sweep — time-to-first-token (ms):")
print(f"  {'variant':<16}" + "".join(f"{'ISL='+str(s):>10}" for s in ISL_SWEEP))
for v in ORDER:
    print(f"  {v:<16}" + "".join(f"{isl[v].get(s, float('nan')):>10.0f}" for s in ISL_SWEEP))

# ---- charts ----
present = [v for v in ORDER if osl.get(v)]
fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))

# 1) energy per decision
pe = [v for v in ORDER if v in energy]
ax[0].bar([LABEL[v] for v in pe], [energy[v]["j"] for v in pe], color=[COLOR[v] for v in pe])
ax[0].set_ylabel("energy per decision (J, total board)"); ax[0].set_title("Energy per decision — battery life")
for i, v in enumerate(pe):
    ax[0].text(i, energy[v]["j"], f"{energy[v]['j']:.1f} J\n{energy[v]['dec_per_wh']:.0f}/Wh", ha="center", va="bottom", fontsize=8)
ax[0].margins(y=0.15)

# 2) reasoning-budget sweep
for v in present:
    xs = [o for o in OSL_SWEEP if o in osl[v]]
    ax[1].plot(xs, [osl[v][o] for o in xs], marker="o", lw=2, label=LABEL[v].replace("\n", " "), color=COLOR[v])
ax[1].set_xlabel("output tokens (reasoning budget)"); ax[1].set_ylabel("end-to-end latency (ms)")
ax[1].set_title("Latency vs reasoning length"); ax[1].set_xticks(OSL_SWEEP); ax[1].grid(alpha=0.3); ax[1].legend(fontsize=8)

# 3) responsiveness — TTFT vs input context
for v in present:
    xs = [s for s in ISL_SWEEP if s in isl[v]]
    ax[2].plot(xs, [isl[v][s] for s in xs], marker="s", lw=2, label=LABEL[v].replace("\n", " "), color=COLOR[v])
ax[2].set_xlabel("input tokens (sensor / history context)"); ax[2].set_ylabel("time-to-first-token (ms)")
ax[2].set_title("Responsiveness: TTFT vs input context"); ax[2].set_xticks(ISL_SWEEP); ax[2].grid(alpha=0.3); ax[2].legend(fontsize=8)

plt.tight_layout(); plt.show()


**What the three charts tell a roboticist:**

- **Energy per decision** — the pruned + distilled + NVFP4 model draws roughly the **same board power** as the others but finishes each decision sooner, so it spends **markedly less energy per inference and gets the most decisions per Wh**. Most of that win is the *pruning/distillation* (fewer layers → less work); NVFP4 adds a further edge on Blackwell's FP4 cores. This is the headline for an untethered robot.
- **Latency vs reasoning length** — at a *reflex* budget (OSL≈8) every variant answers in ~100 ms; as you let it reason longer the latency climbs roughly linearly, and **distilled+NVFP4 stays lowest**, so it can afford a bit more reasoning inside the same loop-time budget. The practical lesson: **budget output tokens** — short for the fast loop, longer only when the robot can pause to deliberate.
- **Responsiveness (TTFT)** — time-to-first-token is **nearly flat** from 128→1024 input tokens, so adding cameras / history context is cheap; **decode, not prefill, is the bottleneck** on Thor. A streaming controller starts acting almost as soon as the request lands.

---
## 8. Wrap-up

We took the DGX-trained, pruned + distilled Cosmos-Reason 2 and made it an **edge-ready artifact on Thor AGX**:

1. Stood up a **Thor-native `aarch64` toolchain** on a prebuilt L4T/Thor vLLM base — no NeMo, no dependency rediscovery ([`Dockerfile.thor`](Dockerfile.thor)).
2. **Quantized to NVFP4** with NVIDIA's best-practice layer selection applied automatically (`lm_head` + vision tower excluded) for both the original and the distilled model — and the §6 data confirmed the **default recipe** is the right call (4-bit costs only noise-level accuracy).
3. **Compiled + served** on Thor via vLLM `torch.compile` + CUDA graphs on Blackwell.
4. **Evaluated quality** (judge-free BLINK + RealWorldQA) and **robotics performance** (energy / reasoning-budget / responsiveness at batch-1), confirming the shippable edge artifact: **distilled + NVFP4** — within noise of baseline quality, lowest latency, and the most decisions per Wh.

### Taking it further: TensorRT-LLM engine build

vLLM compiled and served the NVFP4 checkpoint with no extra steps, which is why we used it. For the most aggressive edge latency, build a **TensorRT-LLM engine** from the *same* ModelOpt NVFP4 checkpoint — TRT-LLM consumes the unified HF quant format directly:

```bash
# (TRT-LLM is built from source on Thor/Tegra — no prebuilt aarch64 wheel yet.)
trtllm-build --checkpoint_dir $BASE_PATH/Cosmos-Reason2-distilled-nvfp4 \
             --output_dir      $BASE_PATH/engines/distilled-nvfp4 \
             --gemm_plugin auto --max_batch_size 8
trtllm-serve $BASE_PATH/engines/distilled-nvfp4 --host 0.0.0.0 --port 8000
```